# 🔧 Full Fine-Tuning: Updating Every Parameter

In the [previous notebook](../01_what_is_fine_tuning.ipynb), we learned what fine-tuning is. Now let's dive into the most straightforward approach: **full fine-tuning**.

## 🎯 What You'll Learn
- What "updating all parameters" actually means
- How gradient descent works (the engine behind training)
- Why full fine-tuning is expensive (and how expensive)
- The danger of "catastrophic forgetting"
- When to use full fine-tuning vs. cheaper methods

## 📋 Prerequisites
- Completed Notebook 01: What is Fine-Tuning?
- Basic Python knowledge

---
## 1️⃣ What Does "Full" Fine-Tuning Mean?

Remember from Notebook 01 that a model is made up of **parameters** (tiny numbers, like knobs on a radio). A model like LLaMA-7B has **7 billion** of these parameters.

**Full fine-tuning** means: we update **every single one** of those 7 billion parameters.

```
  Full Fine-Tuning:
  ═════════════════
  
  Parameter 1:         0.0523  →  adjusts  →  0.0531
  Parameter 2:        -0.1247  →  adjusts  → -0.1239
  Parameter 3:         0.0891  →  adjusts  →  0.0888
  ...                                             
  Parameter 6,999,999,998:  0.0042  →  adjusts  →  0.0045
  Parameter 6,999,999,999: -0.0891  →  adjusts  → -0.0894
  Parameter 7,000,000,000:  0.0213  →  adjusts  →  0.0210
  
  ALL 7 billion get updated! Every. Single. One.
```

### 🏠 The House Renovation Analogy

Think of full fine-tuning as **renovating your entire house**:

```
  Full Fine-Tuning = Full Renovation
  
  ┌──────────────────────────┐      ┌──────────────────────────┐
  │  Before:                 │      │  After:                  │
  │                          │      │                          │
  │  ┌──────┐  ┌──────┐     │      │  ┌──────┐  ┌──────┐     │
  │  │Living│  │Dining│     │      │  │ Exam │  │Patient│    │
  │  │ Room │  │ Room │     │      │  │ Room │  │ Room  │    │
  │  └──────┘  └──────┘     │      │  └──────┘  └──────┘     │
  │  ┌──────┐  ┌──────┐     │      │  ┌──────┐  ┌──────┐     │
  │  │Bed   │  │Bath  │     │      │  │ Lab  │  │Office│     │
  │  │ room │  │ room │     │      │  │      │  │      │     │
  │  └──────┘  └──────┘     │      │  └──────┘  └──────┘     │
  │                          │      │                          │
  │  Regular House           │      │  Medical Clinic           │
  └──────────────────────────┘      └──────────────────────────┘
  
  Every room changed! Expensive, but perfectly customized.
```

Compare this to LoRA (which we'll learn later), which is like **adding an extension** to the house while keeping the original rooms intact.

---
## 2️⃣ How Does It Actually Work? (Gradient Descent)

The engine behind full fine-tuning is called **gradient descent**. Don't let the fancy name scare you - it's actually a simple idea!

### 🏔️ The Mountain Analogy

Imagine you're blindfolded on a mountain and need to reach the lowest valley (the "best" set of parameters). You can't see, but you CAN feel the slope under your feet.

```
  You are here (blindfolded)
        ↓
       YOU
      / ⛰️ \
     /       \
    /    ⛰️   \
   /   /    \  \
  /   /      \  \
 /   /  GOAL  \  \
/___/____⭐____\__\___
         ↑
    Lowest point = best parameters
```

**Gradient descent strategy:**
1. Feel which direction goes downhill (compute the **gradient**)
2. Take a small step downhill (update the **parameters**)
3. Repeat until you reach the bottom!

The **gradient** tells you: "This way is downhill!" And the **learning rate** controls how big each step is.

```
  Learning Rate Too Big:     Learning Rate Too Small:     Just Right:
                 
      JUMP!                       tiny step                 Nice step
       ╱→  ←╲                      →→                        →→→
      ╱      ╲                   →→→→                      →→→→→
     ╱        ╲                →→→→→→                    →→→→→→→→
    ╱    ⭐    ╲             →→→→→→→→                  →→→→→→⭐
   (misses goal!)          (takes forever!)            (finds goal!)
```

In [ ]:
# 🏔️ Let's VISUALIZE gradient descent!
# We'll show how parameters get adjusted step by step.

import numpy as np
import matplotlib.pyplot as plt

# Create a simple "loss landscape" (the mountain)
# In reality this has billions of dimensions, but we'll use 2D for visualization

def loss_function(x, y):
    """A simple bowl-shaped loss function. The minimum is at (2, 1)."""
    return (x - 2)**2 + (y - 1)**2 + 0.5 * np.sin(3*x) * np.cos(3*y)

# Create grid for contour plot
x = np.linspace(-2, 6, 100)
y = np.linspace(-3, 5, 100)
X, Y = np.meshgrid(x, y)
Z = loss_function(X, Y)

# Simulate gradient descent
# Start far from the optimum
param_x, param_y = 5.0, 4.0  # Starting parameters
learning_rate = 0.1
path_x, path_y = [param_x], [param_y]

for step in range(30):
    # Compute gradient (which direction is downhill?)
    grad_x = 2 * (param_x - 2) + 1.5 * np.cos(3*param_x) * np.cos(3*param_y)
    grad_y = 2 * (param_y - 1) - 1.5 * np.sin(3*param_x) * np.sin(3*param_y)
    
    # Take a step downhill (update parameters!)
    param_x -= learning_rate * grad_x
    param_y -= learning_rate * grad_y
    
    path_x.append(param_x)
    path_y.append(param_y)

# Plot!
fig, ax = plt.subplots(figsize=(10, 8))

# Contour plot (shows the "mountain" from above)
contour = ax.contourf(X, Y, Z, levels=20, cmap='RdYlGn_r', alpha=0.8)
ax.contour(X, Y, Z, levels=20, colors='gray', alpha=0.3, linewidths=0.5)
plt.colorbar(contour, ax=ax, label='Loss (lower = better)')

# Plot the path of gradient descent
ax.plot(path_x, path_y, 'b-o', markersize=4, linewidth=2, 
        label='Gradient descent path', color='#1565C0', zorder=5)
ax.plot(path_x[0], path_y[0], 'rs', markersize=15, label='Start (random parameters)',
        zorder=6)
ax.plot(path_x[-1], path_y[-1], 'g*', markersize=20, label='End (optimized parameters)',
        zorder=6)

# Add step numbers
for i in range(0, len(path_x), 5):
    ax.annotate(f'Step {i}', (path_x[i], path_y[i]),
               xytext=(5, 5), textcoords='offset points',
               fontsize=8, color='blue')

ax.set_xlabel('Parameter 1 (one of billions)', fontsize=12)
ax.set_ylabel('Parameter 2 (another of billions)', fontsize=12)
ax.set_title('Gradient Descent: Finding the Best Parameters\n'
             '(Each step adjusts parameters to reduce loss)', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.savefig('gradient_descent.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Started at parameters: ({path_x[0]:.1f}, {path_y[0]:.1f})")
print(f"Ended at parameters:   ({path_x[-1]:.2f}, {path_y[-1]:.2f})")
print(f"The model 'walked downhill' to find better parameters!")

### 🔄 The Full Fine-Tuning Loop

Here's what happens in each step of training:

```
  For each batch of training examples:
  
  ┌─────────────────────────────────────────────────────────┐
  │  1. FORWARD PASS                                        │
  │     Feed input through the model → get prediction       │
  │     "Patient has fever" → model predicts → "Common cold" │
  └─────────────────────────┬───────────────────────────────┘
                            │
                            v
  ┌─────────────────────────────────────────────────────────┐
  │  2. COMPUTE LOSS                                        │
  │     Compare prediction to correct answer                │
  │     Model said: "Common cold"                           │
  │     Correct answer: "Flu"                               │
  │     Loss = How wrong was it? (a number)                 │
  └─────────────────────────┬───────────────────────────────┘
                            │
                            v
  ┌─────────────────────────────────────────────────────────┐
  │  3. BACKWARD PASS (Backpropagation)                     │
  │     Calculate: "Which parameters caused the error?"     │
  │     This gives us the GRADIENT for each parameter       │
  └─────────────────────────┬───────────────────────────────┘
                            │
                            v
  ┌─────────────────────────────────────────────────────────┐
  │  4. UPDATE PARAMETERS                                   │
  │     Nudge ALL parameters in the direction that           │
  │     reduces the error                                   │
  │     parameter = parameter - learning_rate × gradient    │
  └─────────────────────────────────────────────────────────┘
  
  Repeat thousands of times until the model gets good!
```

In [ ]:
# 🎓 Let's see the full fine-tuning loop in action!
# We'll fine-tune a simple neural network to classify data.

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ── Create some simple training data ──
# Task: Classify points as "Category A" (above the line) or "Category B" (below)
# This simulates having a specific task we want the model to learn

n_samples = 200
X = np.random.randn(n_samples, 2)  # 200 points with 2 features

# True boundary: y = 0.5*x + 0.3 (a line)
y = (X[:, 1] > 0.5 * X[:, 0] + 0.3).astype(float)  # 0 or 1

# Split into training and validation
X_train, X_val = X[:160], X[160:]
y_train, y_val = y[:160], y[160:]

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Features per sample: {X_train.shape[1]}")
print(f"\nThis is like having {len(X_train)} labeled examples for our task!")

In [ ]:
# ── Define a simple neural network ──
# This represents our "pre-trained model" - just with random starting weights
# In reality, a pre-trained model starts with GOOD weights from prior training

class SimpleNeuralNetwork:
    def __init__(self, input_size=2, hidden_size=8, output_size=1):
        """A tiny neural network with one hidden layer.
        
        Think of it as a mini-model with:
        - input_size: how many features each example has
        - hidden_size: how many "neurons" in the middle
        - output_size: how many outputs (1 for binary classification)
        """
        # Initialize ALL parameters (weights)
        # In a real LLM, there would be BILLIONS of these
        self.W1 = np.random.randn(input_size, hidden_size) * 0.5  # Layer 1 weights
        self.b1 = np.zeros(hidden_size)                            # Layer 1 biases
        self.W2 = np.random.randn(hidden_size, output_size) * 0.5 # Layer 2 weights
        self.b2 = np.zeros(output_size)                            # Layer 2 biases
        
        # Count total parameters
        self.n_params = (
            self.W1.size + self.b1.size + 
            self.W2.size + self.b2.size
        )
    
    def sigmoid(self, z):
        """Squashes any number to between 0 and 1.
        Used to get a probability-like output."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def forward(self, X):
        """Forward pass: input → prediction."""
        # Layer 1: input → hidden
        self.z1 = X @ self.W1 + self.b1      # Matrix multiplication + bias
        self.a1 = np.maximum(0, self.z1)       # ReLU activation (keep positive values)
        
        # Layer 2: hidden → output
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.sigmoid(self.z2)        # Sigmoid to get probability
        
        return self.a2
    
    def compute_loss(self, y_pred, y_true):
        """How wrong is the model? (Binary cross-entropy loss)."""
        epsilon = 1e-8  # Tiny number to avoid log(0)
        y_true = y_true.reshape(-1, 1)
        loss = -np.mean(
            y_true * np.log(y_pred + epsilon) + 
            (1 - y_true) * np.log(1 - y_pred + epsilon)
        )
        return loss
    
    def backward(self, X, y_true, learning_rate=0.01):
        """Backward pass: compute gradients and update ALL parameters.
        
        This is the FULL fine-tuning part - every weight gets adjusted!
        """
        m = X.shape[0]  # Number of samples
        y_true = y_true.reshape(-1, 1)
        
        # ── Compute gradients (which direction to adjust each parameter) ──
        # Layer 2 gradients
        dz2 = self.a2 - y_true
        dW2 = (self.a1.T @ dz2) / m
        db2 = np.mean(dz2, axis=0)
        
        # Layer 1 gradients
        dz1 = (dz2 @ self.W2.T) * (self.z1 > 0)  # ReLU derivative
        dW1 = (X.T @ dz1) / m
        db1 = np.mean(dz1, axis=0)
        
        # ── UPDATE ALL PARAMETERS ──
        # This is what makes it "FULL" fine-tuning!
        self.W2 -= learning_rate * dW2  # Update weight 1
        self.b2 -= learning_rate * db2  # Update bias 1
        self.W1 -= learning_rate * dW1  # Update weight 2
        self.b1 -= learning_rate * db1  # Update bias 2
        # Every single parameter was updated! ^

# Create the model
model = SimpleNeuralNetwork()
print(f"Our tiny model has {model.n_params} parameters.")
print(f"\nIn comparison:")
print(f"  BERT-base:   110,000,000 parameters")
print(f"  LLaMA-7B:  7,000,000,000 parameters")
print(f"  GPT-4:     ~1,700,000,000,000 parameters")
print(f"\nFull fine-tuning updates ALL of them!")

In [ ]:
# ── Let's run the full fine-tuning training loop! ──

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

n_epochs = 100
learning_rate = 0.5

print("Training (Full Fine-Tuning)...\n")
print(f"{'Epoch':>6} | {'Train Loss':>11} | {'Val Loss':>11} | {'Train Acc':>10} | {'Val Acc':>10}")
print("-" * 62)

for epoch in range(n_epochs):
    # ── Forward pass ──
    y_pred_train = model.forward(X_train)
    train_loss = model.compute_loss(y_pred_train, y_train)
    
    # ── Backward pass (update ALL parameters) ──
    model.backward(X_train, y_train, learning_rate)
    
    # ── Evaluate on validation set ──
    y_pred_val = model.forward(X_val)
    val_loss = model.compute_loss(y_pred_val, y_val)
    
    # ── Calculate accuracy ──
    train_acc = np.mean((y_pred_train.flatten() > 0.5) == y_train)
    val_acc = np.mean((y_pred_val.flatten() > 0.5) == y_val)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    # Print every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"{epoch+1:>6} | {train_loss:>11.4f} | {val_loss:>11.4f} | {train_acc:>9.1%} | {val_acc:>9.1%}")

print(f"\nFinal accuracy: {val_acc:.1%} on validation data!")

In [ ]:
# 📊 Visualize the training progress

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Loss curves ──
ax = axes[0]
ax.plot(train_losses, label='Training Loss', color='#1565C0', linewidth=2)
ax.plot(val_losses, label='Validation Loss', color='#C62828', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (lower = better)')
ax.set_title('Loss Over Training', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# ── 2. Accuracy curves ──
ax = axes[1]
ax.plot(train_accuracies, label='Training Accuracy', color='#1565C0', linewidth=2)
ax.plot(val_accuracies, label='Validation Accuracy', color='#C62828', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (higher = better)')
ax.set_title('Accuracy Over Training', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# ── 3. Decision boundary ──
ax = axes[2]

# Create mesh for decision boundary
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = model.forward(grid).reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

# Plot data points
ax.scatter(X_val[y_val == 0, 0], X_val[y_val == 0, 1], 
          c='red', marker='o', edgecolors='black', s=60, label='Category A', zorder=5)
ax.scatter(X_val[y_val == 1, 0], X_val[y_val == 1, 1], 
          c='blue', marker='s', edgecolors='black', s=60, label='Category B', zorder=5)

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Learned Decision Boundary', fontweight='bold')
ax.legend()

plt.suptitle('Full Fine-Tuning Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('full_fine_tuning_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("All parameters were updated during training!")
print(f"The model learned to separate the two categories with {val_acc:.1%} accuracy.")

---
## 3️⃣ The Cost Problem: Why Full Fine-Tuning is Expensive

Our tiny model had only 33 parameters. Real models have **billions**. This creates serious resource requirements.

### 💾 Memory Requirements

During full fine-tuning, the GPU needs to store:

```
  What needs to fit in GPU memory:
  ═════════════════════════════════
  
  1. The model itself            (all parameters)
  2. The gradients               (one gradient per parameter)
  3. The optimizer state         (2x parameters for Adam optimizer)
  4. Activations                 (intermediate calculations)
  
  Rule of thumb: You need roughly 4x the model size in memory!
  
  ┌──────────────────────────────────────────────────────┐
  │  Model Size    GPU Memory Needed   GPU Required      │
  │  ──────────    ────────────────     ────────────      │
  │  1B params     ~16 GB              1x A100 (40GB)    │
  │  7B params     ~112 GB             3x A100 (80GB)    │
  │  13B params    ~208 GB             6x A100 (80GB)    │
  │  70B params    ~1,120 GB           14x A100 (80GB)   │
  └──────────────────────────────────────────────────────┘
  
  That's A LOT of expensive GPUs! 💰💰💰
```

### 🤔 Why So Much Memory?

Think of it like moving furniture:

```
  Storing a couch (the parameter):           Takes space
  + A plan for where to move it (gradient):  More space  
  + Memory of recent moves (optimizer):      Even more space
  + The path to walk (activations):          Yet more space
  ────────────────────────────────────────────────────────
  Total: About 4x the couch size!            A lot of space!
```

In [ ]:
# 📊 Visualize the memory requirements

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── 1. Memory breakdown ──
ax = axes[0]
categories = ['Model\nWeights', 'Gradients', 'Optimizer\nState', 'Activations']
# For a 7B model in FP16
memory_gb = [14, 14, 56, 28]  # Approximate GB
colors = ['#1565C0', '#F44336', '#FF9800', '#4CAF50']

bars = ax.bar(categories, memory_gb, color=colors, edgecolor='black', linewidth=1.5)

for bar, mem in zip(bars, memory_gb):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{mem} GB', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('GPU Memory (GB)', fontsize=12)
ax.set_title('Memory Needed for Full Fine-Tuning\n(7B parameter model)', 
             fontsize=14, fontweight='bold')
ax.axhline(y=80, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax.text(3.5, 82, 'A100 GPU limit (80GB)', color='red', fontsize=10, ha='right')
ax.set_ylim(0, 70)
ax.grid(True, alpha=0.3, axis='y')

total = sum(memory_gb)
ax.text(0.5, 0.95, f'Total: ~{total} GB needed!', transform=ax.transAxes,
        fontsize=13, fontweight='bold', ha='center', va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

# ── 2. Cost comparison ──
ax = axes[1]
methods = ['Train from\nScratch', 'Full\nFine-Tuning', 'LoRA', 'QLoRA']
costs = [100, 25, 3, 1]
gpu_hours = ['100,000+\nGPU hours', '1,000+\nGPU hours', '~50\nGPU hours', '~10\nGPU hours']
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3']

bars = ax.barh(methods, costs, color=colors, edgecolor='black', linewidth=1.5)

for bar, hours in zip(bars, gpu_hours):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2.,
            hours, ha='left', va='center', fontsize=9)

ax.set_xlabel('Relative Cost', fontsize=12)
ax.set_title('Cost Comparison of Training Methods\n(7B parameter model)', 
             fontsize=14, fontweight='bold')
ax.set_xlim(0, 130)

plt.tight_layout()
plt.savefig('cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Full fine-tuning is expensive, but it's MUCH cheaper than training from scratch!")
print("And LoRA/QLoRA (next notebook!) makes it even cheaper.")

---
## 4️⃣ Catastrophic Forgetting ⚠️

There's a famous problem with full fine-tuning called **catastrophic forgetting**.

### 🧠 What Is It?

When you fine-tune a model on a new task, it can **forget** what it previously knew!

```
  CATASTROPHIC FORGETTING:
  ════════════════════════
  
  Before fine-tuning:                After fine-tuning on medical data:
  ┌──────────────────────┐          ┌──────────────────────┐
  │  ✅ Knows history    │          │  ❌ Forgot history   │
  │  ✅ Knows geography  │          │  ❌ Forgot geography │
  │  ✅ Knows grammar    │          │  ⚠️ Grammar is shaky │
  │  ✅ Can do math      │          │  ❌ Forgot math      │
  │  ❌ No medical skill │          │  ✅ Great at medicine │
  └──────────────────────┘          └──────────────────────┘
  
  It learned medicine but forgot everything else!
```

### 🎓 The Student Analogy

Imagine a student who studies ONLY math for a year and nothing else:
- They become amazing at math!
- But they forget history, science, and literature
- Their brain "overwrote" old knowledge with new knowledge

### 🛡️ How to Prevent Catastrophic Forgetting

| Strategy | How It Works | Analogy |
|----------|-------------|--------|
| **Low learning rate** | Make tiny adjustments | Gentle editing vs. rewriting |
| **Mix in general data** | Include some original training data | Study math AND review other subjects |
| **Early stopping** | Don't train too long | Know when to stop studying |
| **Use LoRA instead** | Only add new parameters, don't change old ones | Add a sticky note instead of erasing the textbook |
| **Regularization** | Penalize big parameter changes | Don't stray too far from original values |

In [ ]:
# 📊 Simulate catastrophic forgetting

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
epochs = np.arange(0, 50)

# Simulate performance on different tasks during fine-tuning
# The model gets better at the new task but worse at old tasks

# New task performance (medical diagnosis)
new_task = 50 + 45 * (1 - np.exp(-0.1 * epochs)) + np.random.normal(0, 1, len(epochs))
new_task = np.clip(new_task, 0, 100)

# Old task 1 (general knowledge) - degrades with aggressive fine-tuning
old_task1 = 90 - 35 * (1 - np.exp(-0.06 * epochs)) + np.random.normal(0, 1.5, len(epochs))

# Old task 2 (grammar) - degrades more slowly
old_task2 = 85 - 20 * (1 - np.exp(-0.04 * epochs)) + np.random.normal(0, 1, len(epochs))

# Old task 3 (math) - degrades significantly
old_task3 = 80 - 40 * (1 - np.exp(-0.08 * epochs)) + np.random.normal(0, 1.5, len(epochs))

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(epochs, new_task, 'g-', linewidth=3, label='New Task (Medical Diagnosis)', 
        color='#2E7D32')
ax.plot(epochs, old_task1, '--', linewidth=2, label='Old Task: General Knowledge', 
        color='#F44336')
ax.plot(epochs, old_task2, '--', linewidth=2, label='Old Task: Grammar', 
        color='#FF9800')
ax.plot(epochs, old_task3, '--', linewidth=2, label='Old Task: Math', 
        color='#9C27B0')

# Mark the danger zone
ax.axvspan(25, 50, alpha=0.08, color='red')
ax.text(37, 95, 'DANGER ZONE\n(Forgetting old skills!)', fontsize=11,
        ha='center', color='red', fontweight='bold', alpha=0.7)

# Mark the sweet spot
ax.axvline(x=15, color='green', linestyle=':', linewidth=2, alpha=0.5)
ax.annotate('Sweet spot: Good at\nnew task, still\nremembers old tasks', 
           xy=(15, 75), xytext=(22, 75),
           fontsize=10, color='green', fontweight='bold',
           arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.set_xlabel('Fine-Tuning Epochs', fontsize=12)
ax.set_ylabel('Performance (%)', fontsize=12)
ax.set_title('Catastrophic Forgetting: The Model Forgets What It Knew!', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='center left')
ax.grid(True, alpha=0.3)
ax.set_ylim(30, 105)

plt.tight_layout()
plt.savefig('catastrophic_forgetting.png', dpi=150, bbox_inches='tight')
plt.show()

print("As the model gets better at its new task (green line going UP),")
print("it gets WORSE at old tasks (red/orange/purple lines going DOWN).")
print("\nThis is why LoRA was invented - it avoids this problem!")

---
## 5️⃣ When to Use Full Fine-Tuning

Despite its costs, full fine-tuning is still the right choice sometimes:

### ✅ Use Full Fine-Tuning When:

```
  1. You need MAXIMUM quality
     └── Nothing else will do - you need the absolute best performance
     
  2. Your task is VERY different from the original training
     └── E.g., fine-tuning an English model for Japanese medical texts
     
  3. You have LOTS of data (10K+ examples)
     └── Enough data to justify updating all parameters
     
  4. You have the RESOURCES ($$$, GPUs)
     └── Big company with cloud compute budget
     
  5. The model is SMALL enough
     └── Models under 1B parameters are feasible on a single GPU
```

### ❌ Skip Full Fine-Tuning When:

```
  1. Your budget is limited → Use LoRA or QLoRA
  2. You have little data (< 1000 examples) → Risk of overfitting
  3. The model is huge (7B+) → Memory requirements are extreme
  4. You need to maintain original capabilities → Use LoRA
  5. You're experimenting → LoRA is faster to iterate with
```

### 📊 Decision Matrix

| Factor | Full Fine-Tuning | LoRA/QLoRA |
|--------|:---:|:---:|
| Quality | Best possible | Nearly as good (95-99%) |
| Cost | High | Low |
| Speed | Slow | Fast |
| Memory | Very high | Low |
| Risk of forgetting | High | Low |
| Ease of use | Moderate | Easy |
| Best for | Critical production models | Most use cases |

---
## 🎯 Key Takeaways

1. **Full fine-tuning updates ALL parameters** in the model

2. **Gradient descent** is the algorithm that adjusts parameters (walk downhill on the loss landscape)

3. **It's expensive** - you need ~4x the model size in GPU memory

4. **Catastrophic forgetting** is a real danger - the model can forget old knowledge

5. **Use it when** you need maximum quality and have the resources

6. **Skip it when** you're on a budget - LoRA is usually good enough!

## 🚀 What's Next?

Now that you understand full fine-tuning and its drawbacks, let's learn about **LoRA** - the clever trick that gives you 95% of the quality at 5% of the cost!

| Next Notebook | What You'll Learn |
|---------------|------------------|
| [Understanding LoRA](../lora-qlora/01_understanding_lora.ipynb) | How to fine-tune efficiently with Low-Rank Adaptation |

---

[Back to Module README](../README.md)